In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision.models import vit_b_16, ViT_B_16_Weights
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import math
import numpy as np
import cv2
import matplotlib.pyplot as plt

In [4]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# download pre-trained model
pretrained_weights = ViT_B_16_Weights.IMAGENET1K_V1
model = vit_b_16(weights=pretrained_weights).to(device)

In [ ]:
# patch the model to provide also attention maps
def layer_forward(self, input: torch.Tensor):
    torch._assert(input.dim() == 3, f"Expected (batch_size, seq_length, hidden_dim) got {input.shape}")
    x = self.ln_1(input)
    x, att_map = self.self_attention(x, x, x, need_weights=True, average_attn_weights=False)
    att_maps.append(att_map)
    x = self.dropout(x)
    x = x + input
    y = self.ln_2(x)
    y = self.mlp(y)
    return x + y

for layer in model.encoder.layers:
    layer.forward = layer_forward.__get__(layer, type(layer))

In [ ]:
import kagglehub
path = kagglehub.dataset_download("ambityga/imagenet100")
print("Path to dataset files:", path)

In [ ]:
# download ImageNet Labels
!wget http://storage.googleapis.com/bit_models/ilsvrc2012_wordnet_lemmas.txt

In [ ]:
imagenet_labels = dict(enumerate(open('ilsvrc2012_wordnet_lemmas.txt')))
print(imagenet_labels)

In [ ]:
!ls -l $path/val.X/n01773549

In [ ]:
imagepath = path + '/val.X/' + 'n01773549' +'/' + 'ILSVRC2012_val_00008316.JPEG'

In [ ]:
import cv2
image = cv2.imread(imagepath)
from google.colab.patches import cv2_imshow
cv2_imshow(image)

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5, 0.5, 0.5),
        std=(0.5, 0.5, 0.5)
    )
])

In [ ]:
size = (224, 224)

In [ ]:
blob = transform(cv2.resize(cv2.cvtColor(image,cv2.COLOR_BGR2RGB),size)).unsqueeze(0).to(device)

In [ ]:
blob.shape

In [ ]:
with torch.no_grad():
    att_maps = []
    logits = model(blob)

In [ ]:
# Present probabilities of categories
probs = torch.sigmoid(logits)[0]
labels = torch.argsort(probs, dim=-1, descending=True)
top5 = labels[:5]
print("Prediction:")
for idx in top5:
    print(f'{probs[idx.item()]:.5f} : {imagenet_labels[idx.item()]}', end='')

In [ ]:
att_maps = torch.cat(att_maps, dim=0)

In [ ]:
att_maps.shape

In [ ]:
def draw_mask(img, mask):
    H, W = img.shape[:2]
    mask_resized = cv2.resize(mask, (W, H), interpolation=cv2.INTER_LINEAR)
    if img.ndim == 3:
        mask_resized = np.repeat(mask_resized[:, :, None], 3, axis=2)
    result = (img.astype(np.float32) * mask_resized).clip(0, 255).astype(np.uint8)
    return result

In [ ]:
base = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
num_layers, num_heads, N, _ = att_maps.shape
patch_size = int(math.sqrt(size[0]*size[1]//(N-1)))
att_size = (size[0]//patch_size, size[1]//patch_size)
plt.figure(figsize=(2 * num_heads, 2 * num_layers))
for t in range(num_layers):
    for i in range(num_heads):
        head = att_maps[t, i]
        mask = head[0, 1:].reshape(att_size[0], att_size[1])
        mask = mask.detach().cpu().numpy()
        disp = draw_mask(base, mask)
        plt.subplot(num_layers, num_heads, t * num_heads + i + 1)
        plt.imshow(disp, cmap='gray')
        plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# joint_attention pre layer
att = att_maps.mean(dim=1)
print(att.shape)

In [ ]:
# visualize averaged and normalizes heads
plt.figure(figsize=(num_layers,2))
for t in range(num_layers):
    head = att[t]
    mask = head[0, 1:].reshape(att_size[0], att_size[1])
    mask = mask.detach().cpu().numpy()
    disp = draw_mask(base, mask)
    plt.subplot(1, num_layers, t+1)
    plt.imshow(disp, cmap='gray')
    plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# re-train the model for MNIST

In [5]:
# adjust the model for 10 categories of MNIST dataset
pretrained_weights = ViT_B_16_Weights.IMAGENET1K_V1
model = vit_b_16(weights=pretrained_weights).to(device)
num_classes = 10
model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)
model = model.to(device)

In [6]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=pretrained_weights.transforms().mean,
        std=pretrained_weights.transforms().std
    ),
])

In [7]:
# get datasets
train_dataset = datasets.MNIST(root='data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='data', train=False, download=True, transform=transform)

100%|██████████| 9.91M/9.91M [00:01<00:00, 4.99MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 128kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.25MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 13.6MB/s]


In [16]:
# create train and test dataloader
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [9]:
def train(num_epochs):
    global epochs
    for _ in range(num_epochs):
        epochs += 1
        # train
        model.train()
        total_loss = 0
        correct = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            correct += (outputs.argmax(1) == labels).sum().item()
        train_loss = total_loss / len(train_loader)
        train_acc = correct / len(train_loader.dataset)
        # test
        model.eval()
        correct = 0
        for images, labels in test_loader:
            with torch.no_grad():
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
            correct += (outputs.argmax(1) == labels).sum().item()
        test_acc = correct / len(test_loader.dataset)
        print(f"Epoch [{epochs}] Loss: {train_loss:.4f} Train Acc: {train_acc:.4f} Test Acc: {test_acc:.4f}")

In [10]:
# training
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.heads.head.parameters(), lr=0.001)
epochs = 0
for param in model.parameters():
    param.requires_grad = False
for param in model.heads.head.parameters():
    param.requires_grad = True
train(10)

Epoch [1] Loss: 0.5934 Train Acc: 0.8712 Test Acc: 0.9458
Epoch [2] Loss: 0.2133 Train Acc: 0.9473 Test Acc: 0.9576
Epoch [3] Loss: 0.1623 Train Acc: 0.9569 Test Acc: 0.9614
Epoch [4] Loss: 0.1378 Train Acc: 0.9621 Test Acc: 0.9645
Epoch [5] Loss: 0.1228 Train Acc: 0.9657 Test Acc: 0.9670
Epoch [6] Loss: 0.1123 Train Acc: 0.9680 Test Acc: 0.9690
Epoch [7] Loss: 0.1044 Train Acc: 0.9699 Test Acc: 0.9701
Epoch [8] Loss: 0.0982 Train Acc: 0.9715 Test Acc: 0.9712
Epoch [9] Loss: 0.0931 Train Acc: 0.9727 Test Acc: 0.9723
Epoch [10] Loss: 0.0888 Train Acc: 0.9736 Test Acc: 0.9735


In [15]:
# fine tuning
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001)
epochs = 0
for param in model.parameters():
    param.requires_grad = True
train(10)

OutOfMemoryError: CUDA out of memory. Tried to allocate 38.00 MiB. GPU 0 has a total capacity of 39.56 GiB of which 18.88 MiB is free. Process 32563 has 39.53 GiB memory in use. Of the allocated memory 38.92 GiB is allocated by PyTorch, and 118.87 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# get few samples
test_iter = iter(test_loader)
x_sample, y_sample = next(test_iter)
print(x_sample.shape)

In [ ]:
# use the model on the samples
input_images = x_sample[0:100].to(device)
output_logits = model(input_images)
output_categories = [ category.item() for category in torch.argmax(output_logits.to('cpu'), dim=1) ]
print(output_categories)
categories = [ category.item() for category in y_sample ]
print(categories)


In [ ]:
# show results
def render(digit):
    img = np.zeros((28, 28), dtype=np.uint8)
    cv2.putText(img, str(digit), (6, 22), cv2.FONT_HERSHEY_SIMPLEX, 0.9, 255, 2)
    return img

plt.figure(figsize=(20, 40))
for b in range(10):
    for i in range(10):
        input_img = (x_sample[10*b+i].squeeze(0).detach().numpy()*255).astype(np.uint8)
        plt.subplot(20, 10, 20*b+i+1)
        plt.imshow(input_img, cmap='gray')
        plt.axis('off')
        output_img = render(output_categories[10*b+i])
        plt.subplot(20, 10, 20*b+i+1+10)
        plt.imshow(~output_img, cmap='gray')
        plt.axis('off')

plt.show()